## Практична

## завдання 1

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"
df = pd.read_csv(url)

columns = ['book_id', 'title', 'authors', 'average_rating', 'ratings_count']
df = df[columns]

df = df.dropna()

print(f"Розмір датасету після очищення: {df.shape}")
print(df.head())

Розмір датасету після очищення: (10000, 5)
   book_id                                              title  \
0        1            The Hunger Games (The Hunger Games, #1)   
1        2  Harry Potter and the Sorcerer's Stone (Harry P...   
2        3                            Twilight (Twilight, #1)   
3        4                              To Kill a Mockingbird   
4        5                                   The Great Gatsby   

                       authors  average_rating  ratings_count  
0              Suzanne Collins            4.34        4780653  
1  J.K. Rowling, Mary GrandPré            4.44        4602479  
2              Stephenie Meyer            3.57        3866839  
3                   Harper Lee            4.25        3198671  
4          F. Scott Fitzgerald            3.89        2683664  


## завдання 2

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = df.reset_index(drop=True)

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['authors'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def recommend(title, n=5):
    if title not in df['title'].values:
        return f"Книгу '{title}' не знайдено в датасеті."
    
    idx = df[df['title'] == title].index[0]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:n+1]
    
    book_indices = [i[0] for i in sim_scores]
    
    return df[['title', 'authors', 'average_rating']].iloc[book_indices]

print("Рекомендації для 'The Hobbit':")
print(recommend('The Hobbit', n=5))

print("\nРекомендації для 'The Da Vinci Code':")
print(recommend('The Da Vinci Code', n=5))

Рекомендації для 'The Hobbit':
                                                 title         authors  \
18   The Fellowship of the Ring (The Lord of the Ri...  J.R.R. Tolkien   
154         The Two Towers (The Lord of the Rings, #2)  J.R.R. Tolkien   
160  The Return of the King (The Lord of the Rings,...  J.R.R. Tolkien   
188  The Lord of the Rings (The Lord of the Rings, ...  J.R.R. Tolkien   
963  J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...  J.R.R. Tolkien   

     average_rating  
18             4.34  
154            4.42  
160            4.51  
188            4.47  
963            4.59  

Рекомендації для 'The Da Vinci Code':
Книгу 'The Da Vinci Code' не знайдено в датасеті.


## завдання 3

In [4]:
from surprise import Dataset, Reader, accuracy
from surprise.model_selection import train_test_split
from surprise.prediction_algorithms.matrix_factorization import SVD

ratings_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"
ratings_df = pd.read_csv(ratings_url)

print(ratings_df.head())

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings_df[['user_id', 'book_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model = SVD(random_state=42)
model.fit(trainset)

predictions = model.test(testset)
rmse = accuracy.rmse(predictions)

   user_id  book_id  rating
0        1      258       5
1        2     4081       4
2        2      260       5
3        2     9296       5
4        2     2318       3
RMSE: 0.8310


## завдання 4

In [5]:
target_user_id = 1

read_books = ratings_df[ratings_df['user_id'] == target_user_id]['book_id'].tolist()

unread_books = df[~df['book_id'].isin(read_books)].copy()

unread_books['predicted_rating'] = unread_books['book_id'].apply(lambda x: model.predict(target_user_id, x).est)

top_5_recommendations = unread_books.sort_values(by='predicted_rating', ascending=False).head(5)

result_df = top_5_recommendations[['book_id', 'title', 'authors', 'predicted_rating']]

print(f"Топ-5 рекомендацій для користувача ID {target_user_id}:")
print(result_df.to_string(index=False))

result_df.to_csv('book_recommendations.csv', index=False, encoding='utf-8')

Топ-5 рекомендацій для користувача ID 1:
 book_id                                                             title                          authors  predicted_rating
    8946                                                         The Divan                            Hafez          4.748697
    3628                                    The Complete Calvin and Hobbes                   Bill Watterson          4.668988
    6590 The Authoritative Calvin and Hobbes: A Calvin and Hobbes Treasury                   Bill Watterson          4.652345
     852                                           When Breath Becomes Air Paul Kalanithi, Abraham Verghese          4.611733
    7254       Homicidal Psycho Jungle Cat: A Calvin and Hobbes Collection                   Bill Watterson          4.589034


## Домашня

## завдання 1

In [6]:
books_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/books.csv"
ratings_url = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master/ratings.csv"

books_df = pd.read_csv(books_url)
ratings_df = pd.read_csv(ratings_url)

books_df = books_df.dropna().drop_duplicates()
ratings_df = ratings_df.dropna().drop_duplicates()

merged_df = pd.merge(ratings_df, books_df, on='book_id', how='inner')

columns_to_keep = ['book_id', 'title', 'authors', 'user_id', 'rating']
merged_df = merged_df[columns_to_keep]

print(f"Розмір об'єднаного датасету: {merged_df.shape}")
print(merged_df.head())

Розмір об'єднаного датасету: (5311817, 5)
   book_id                                              title  \
0      258  The Shadow of the Wind (The Cemetery of Forgot...   
1     9296  The Drama of the Gifted Child: The Search for ...   
2     2318  The Millionaire Next Door: The Surprising Secr...   
3       26             The Da Vinci Code (Robert Langdon, #2)   
4      315                               Who Moved My Cheese?   

                                 authors  user_id  rating  
0        Carlos Ruiz Zafón, Lucia Graves        1       5  
1               Alice  Miller, Ruth Ward        2       5  
2    Thomas J. Stanley, William D. Danko        2       3  
3                              Dan Brown        2       4  
4  Spencer Johnson, Kenneth H. Blanchard        2       3  


## завдання 2

In [7]:
unique_books = merged_df[['book_id', 'title', 'authors']].drop_duplicates(subset='book_id').reset_index(drop=True)

unique_books['text'] = unique_books['title'] + ' ' + unique_books['authors']

tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(unique_books['text'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

def recommend(title, n=5):
    if title not in unique_books['title'].values:
        print(f"Книгу '{title}' не знайдено.")
        return pd.DataFrame()
    
    idx = unique_books[unique_books['title'] == title].index[0]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:n+1]
    
    book_indices = [i[0] for i in sim_scores]
    
    return unique_books[['book_id', 'title', 'authors']].iloc[book_indices]

recommendations = recommend('The Hobbit', n=5)
print(recommendations)

recommendations.to_csv('content_recommendations.csv', index=False, encoding='utf-8')

      book_id                                              title  \
18        964  J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...   
3007     1129   The History of the Hobbit, Part One: Mr. Baggins   
2443     2309                              The Children of Húrin   
3273      466                          The Hobbit: Graphic Novel   
305       611           The Silmarillion (Middle-Earth Universe)   

                                                authors  
18                                       J.R.R. Tolkien  
3007                   John D. Rateliff, J.R.R. Tolkien  
2443      J.R.R. Tolkien, Christopher Tolkien, Alan Lee  
3273  Chuck Dixon, J.R.R. Tolkien, David Wenzel, Sea...  
305    J.R.R. Tolkien, Christopher Tolkien, Ted Nasmith  


## завдання 3

In [8]:
import pickle

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(merged_df[['user_id', 'book_id', 'rating']], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd_model = SVD(random_state=42)
svd_model.fit(trainset)

predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions)

with open('svd_model.pkl', 'wb') as file:
    pickle.dump(svd_model, file)

RMSE: 0.8326


## завдання 4

In [9]:
with open('svd_model.pkl', 'rb') as file:
    svd_model = pickle.load(file)

unique_books = merged_df[['book_id', 'title', 'authors']].drop_duplicates(subset='book_id').copy()

def recommend_cf(user_id, n=5):
    rated_books = merged_df[merged_df['user_id'] == user_id]['book_id'].tolist()
    
    candidates = unique_books[~unique_books['book_id'].isin(rated_books)].copy()
    
    candidates['predicted_rating'] = candidates['book_id'].apply(lambda book_id: svd_model.predict(user_id, book_id).est)
    
    top_n = candidates.sort_values(by='predicted_rating', ascending=False).head(n)
    top_n.insert(0, 'user_id', user_id)
    
    return top_n[['user_id', 'book_id', 'title', 'authors', 'predicted_rating']]

sample_users = merged_df['user_id'].unique()[:3]
all_recommendations = []

for uid in sample_users:
    user_recs = recommend_cf(uid, n=5)
    print(f"\nТоп-5 рекомендацій для користувача ID {uid}:")
    print(user_recs.to_string(index=False))
    all_recommendations.append(user_recs)

cf_results_df = pd.concat(all_recommendations, ignore_index=True)
cf_results_df.to_csv('cf_results.csv', index=False, encoding='utf-8')


Топ-5 рекомендацій для користувача ID 1:
 user_id  book_id                                                             title                         authors  predicted_rating
       1     3628                                    The Complete Calvin and Hobbes                  Bill Watterson          4.576583
       1     6590 The Authoritative Calvin and Hobbes: A Calvin and Hobbes Treasury                  Bill Watterson          4.546739
       1     8946                                                         The Divan                           Hafez          4.480095
       1     8233                                              The Book of Disquiet Fernando Pessoa, Richard Zenith          4.418110
       1     1010     The Essential Calvin and Hobbes: A Calvin and Hobbes Treasury                  Bill Watterson          4.382649

Топ-5 рекомендацій для користувача ID 2:
 user_id  book_id                                                            title                              